# Lab 05: Model Evaluation
**Week 6 — October 19**

A fictional campus food-delivery app wants to predict **how long an order will take** and to flag orders likely to end in a **refund request**. Before trusting either model, the team needs a fair way to measure it. The data are synthetic and do not describe a real company.

By the end, you should be able to:
- Explain the jobs of training, validation, and test sets.
- Compute and interpret MAE, RMSE, and $R^2$.
- Use cross-validation to choose a regularization strength.
- Compute precision and recall, and explain why accuracy misleads on imbalanced data.
- Choose a classification threshold on validation data.

Work from top to bottom. Run cells with **Shift + Enter**. Replace every **...** in an answer cell and complete the written responses. You can work with a neighbor and ask a tutor/TA for help.

## Setup

Keep **campus_orders.csv** beside this notebook. Each row is one fictional delivery order. The first six columns are known when the order is placed, so they are our features.

| Column | Meaning |
|---|---|
| distance_km | Distance from restaurant to customer (km) |
| num_items | Items in the order |
| restaurant_queue | Orders already waiting at the restaurant |
| courier_trips | Deliveries the courier had completed before |
| dinner_rush | 1 if ordered between 5 and 8 pm |
| raining | 1 if it was raining |
| delivery_minutes | Minutes until delivery (**regression target**) |
| refund_requested | 1 if the customer asked for a refund (**classification target**) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import metrics

import otter

grader = otter.Notebook("Lab05.ipynb", tests_dir="tests/lab05")

orders = pd.read_csv("campus_orders.csv")
features = ["distance_km", "num_items", "restaurant_queue",
            "courier_trips", "dinner_rush", "raining"]
orders.head()

## Part 1: Training, validation, and test sets

| Set | Job | How often |
|---|---|---|
| **Training** | Fit the model's parameters | Every fit |
| **Validation** | Compare models and choose settings | Many times |
| **Test** | One final, honest estimate on new data | **Once**, at the end |

### Q1. Make a 60 / 20 / 20 split

First hold out 20% of all orders as **test**; the other 80% is **dev** (development). Then split **dev** into **train** and **validation** so that they are 60% and 20% of all orders.

**Hint:** In the second split, **test_size** is a fraction of **dev**. What fraction of 80% is 20%? (**stratify** keeps the share of rare refund requests about equal in every set.)

In [ ]:
dev, test = train_test_split(orders, test_size=..., random_state=42,
                             stratify=orders["refund_requested"])
train, validation = train_test_split(dev, test_size=..., random_state=42,
                                     stratify=dev["refund_requested"])
print("Train:", len(train), "| Validation:", len(validation), "| Test:", len(test))

In [ ]:
grader.check("q1")

You should see 1200, 400, and 400 rows.

**Your response:** A teammate tries 50 models and keeps the one with the best **test** score. Why is that score likely too optimistic, and which set should they have used to choose?

> Write your answer here.

## Part 2: Regression metrics

The next cell fits a linear regression to predict **delivery_minutes** on the training rows, then predicts the validation rows. Every scikit-learn model follows this **fit**, then **predict** pattern.

In [ ]:
X_train, y_train = train[features], train["delivery_minutes"]
X_val, y_val = validation[features], validation["delivery_minutes"]
X_test, y_test = test[features], test["delivery_minutes"]

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
val_pred = linear_model.predict(X_val)

### Q2. Compute MAE, RMSE, and $R^2$ by hand

**RMSE** is the square root of MSE, which puts the error back in **minutes**. **$R^2$** compares the model's squared error with that of always predicting the mean $\bar y$: 1 is perfect, 0 is no better than the mean.

$$\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat y_i| \qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i (y_i-\hat y_i)^2} \qquad
R^2 = 1-\frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}$$

Compute all three on the validation rows with NumPy. Save them as **val_mae**, **val_rmse**, and **val_r2**.

**Hint:** Use **np.mean**, **np.abs**, **np.sqrt**, and **np.sum**, with **actual** and **val_pred**.

In [ ]:
actual = y_val.to_numpy()
val_mae = ...
val_rmse = ...
val_r2 = ...

In [ ]:
grader.check("q2")

Run the next cell to compare your values with scikit-learn and with a baseline that always predicts the average training time. The first two columns should match.

In [ ]:
baseline_pred = np.full(len(y_val), y_train.mean())
pd.DataFrame({
    "Your calculation": [val_mae, val_rmse, val_r2],
    "scikit-learn": [metrics.mean_absolute_error(y_val, val_pred),
                     metrics.root_mean_squared_error(y_val, val_pred),
                     metrics.r2_score(y_val, val_pred)],
    "Mean baseline": [metrics.mean_absolute_error(y_val, baseline_pred),
                      metrics.root_mean_squared_error(y_val, baseline_pred),
                      metrics.r2_score(y_val, baseline_pred)],
}, index=["MAE (minutes)", "RMSE (minutes)", "R^2"]).round(3)

**Your response:** Explain the model's validation RMSE in one sentence a delivery worker would understand. Why is the mean baseline's $R^2$ close to 0?

> Write your answer here.

## Part 3: Choosing a regularization budget with cross-validation

The cell below gives the model **polynomial features** (squares and products of the columns), turning 6 columns into 209. That is flexible enough to overfit, so we add **ridge regularization**, which penalizes large weights with strength $\alpha$:

$$\frac{1}{n}\sum_i (y_i - \hat y_i)^2 \;+\; \alpha \sum_j w_j^2$$

You can think of $\alpha$ as setting a **budget** for the weights. A **small $\alpha$** gives a big budget: the model is flexible and can overfit. A **large $\alpha$** gives a small budget: the model is simple and can underfit.

Training error cannot choose $\alpha$, because a bigger budget always fits the training rows better. One validation set could favor an $\alpha$ by luck. **5-fold cross-validation** cuts the dev rows into 5 folds. Each fold takes one turn as validation while the model trains on the other 4. Averaging the 5 errors gives the **CV error**. We pick the $\alpha$ with the lowest CV error. The test set stays untouched.

In [ ]:
X_dev, y_dev = dev[features], dev["delivery_minutes"]

def make_ridge(alpha):
    return make_pipeline(PolynomialFeatures(degree=4, include_bias=False),
                         StandardScaler(),
                         Ridge(alpha=alpha))

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

### Q3. Cross-validate each budget

**cross_val_score** runs the 5 folds for us. It reports MSE as a **negative** number, because scikit-learn treats higher scores as better. For each **alpha**, append the average CV MSE as a **positive** number to **cv_mses**. Then save the alpha with the lowest CV MSE as **best_alpha**.

**Hints:** Use **-np.mean(scores)**. **np.argmin(cv_mses)** gives the *position* of the smallest value; use it to index **alphas**.

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000]
cv_mses = []
for alpha in alphas:
    scores = cross_val_score(make_ridge(alpha), X_dev, y_dev, cv=kfold,
                             scoring="neg_mean_squared_error")
    cv_mses.append(...)

best_alpha = ...
print("CV MSE by alpha:", np.round(cv_mses, 1))
print("Best alpha:", best_alpha)

In [ ]:
grader.check("q3")

Run the cell below after Q3. It plots training and CV error for each alpha, then refits the chosen model on all dev rows and checks it **once** on the test set.

In [ ]:
train_mses = [metrics.mean_squared_error(y_dev, make_ridge(a).fit(X_dev, y_dev).predict(X_dev))
              for a in alphas]
plt.plot(alphas, train_mses, "o-", label="Training MSE")
plt.plot(alphas, cv_mses, "o-", label="5-fold CV MSE")
plt.axvline(best_alpha, color="gray", linestyle="--", label=f"Chosen alpha = {best_alpha}")
plt.xscale("log")
plt.xlabel("alpha (log scale)\n← bigger budget        smaller budget →")
plt.ylabel("MSE (minutes²)")
plt.legend()
plt.show()

final_ridge = make_ridge(best_alpha).fit(X_dev, y_dev)
plain_linear = LinearRegression().fit(X_dev, y_dev)
print("Test RMSE, plain linear regression:",
      round(metrics.root_mean_squared_error(y_test, plain_linear.predict(X_test)), 3))
print("Test RMSE, ridge with chosen alpha:",
      round(metrics.root_mean_squared_error(y_test, final_ridge.predict(X_test)), 3))

### Q4. Interpret the curve

**Your response:** Why does training MSE only go up as alpha increases? Which side of the plot shows overfitting and which shows underfitting? Use the gap between the curves.

> Write your answer here.

## Part 4: Classification metrics and class imbalance

Now we predict **refund_requested**. A refund request (**1**) is the **positive** class. Refund requests are rare, so the data are **imbalanced**.

In [ ]:
y_train_c = train["refund_requested"].to_numpy()
y_val_c = validation["refund_requested"].to_numpy()
y_test_c = test["refund_requested"].to_numpy()

### Q5. The "always say no" model

Save the share of **training** orders with a refund request as **refund_rate**. Then predict **0** for every validation order (**always_no_pred**) and save that model's validation accuracy, the share of predictions that are correct, as **always_no_accuracy**.

**Hints:** The mean of a 0/1 array is the share of 1s. Use **np.zeros(n, dtype=int)** and **np.mean(a == b)**.

In [ ]:
refund_rate = ...
always_no_pred = ...
always_no_accuracy = ...
print("Refund rate:", round(refund_rate, 3), "| Always-no accuracy:", round(always_no_accuracy, 3))

In [ ]:
grader.check("q5")

**Your response:** How many refund requests does this model catch? Why is accuracy misleading when one class is rare?

> Write your answer here.

The next cell fits a **logistic regression**. **predict_proba** gives each order's probability of a refund request. A **threshold** turns probabilities into labels; scikit-learn's default is **0.5**.

In [ ]:
classifier = make_pipeline(StandardScaler(), LogisticRegression())
classifier.fit(train[features], y_train_c)
val_probs = classifier.predict_proba(validation[features])[:, 1]
pred_05 = (val_probs >= 0.5).astype(int)

### Q6. Precision and recall

| | Predicted 1 | Predicted 0 |
|---|---|---|
| **Actually 1** | True positive (**TP**) | False negative (**FN**) |
| **Actually 0** | False positive (**FP**) | True negative (**TN**) |

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}$$

**Precision:** of the orders we flagged, what share really requested a refund? **Recall:** of the real refund requests, what share did we flag?

Using **pred_05**, fill in **fp** and **fn** (**tp** and **tn** are done), then compute **precision_05** and **recall_05**.

In [ ]:
tp = np.sum((pred_05 == 1) & (y_val_c == 1))
tn = np.sum((pred_05 == 0) & (y_val_c == 0))
fp = ...
fn = ...

precision_05 = ...
recall_05 = ...

print("TP:", tp, "| FP:", fp, "| FN:", fn, "| TN:", tn)
print("Accuracy:", round((tp + tn) / len(y_val_c), 3),
      "| Precision:", round(precision_05, 3), "| Recall:", round(recall_05, 3))

In [ ]:
grader.check("q6")

**Your response:** Compare this model's accuracy with Q5's. Describe its recall in plain words ("Of all the orders that ..."). Is a 0.5 threshold useful for a support team that wants to reach unhappy customers?

> Write your answer here.

## Part 5: Choosing a threshold

Lowering the threshold flags more orders. That usually raises recall but lowers precision. Support decides that missing an unhappy customer is worse than sending an unneeded apology coupon. Their goal is to **catch at least 70% of refund requests (recall ≥ 0.70) while flagging as few orders as possible**, which means using the **highest** threshold that meets the goal.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.55, 0.05), 2)
threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "orders_flagged": [(val_probs >= t).sum() for t in thresholds],
    "precision": [metrics.precision_score(y_val_c, val_probs >= t, zero_division=0) for t in thresholds],
    "recall": [metrics.recall_score(y_val_c, val_probs >= t) for t in thresholds],
}).round(3)

plt.plot(threshold_table["threshold"], threshold_table["precision"], "o-", label="Precision")
plt.plot(threshold_table["threshold"], threshold_table["recall"], "o-", label="Recall")
plt.axhline(0.70, color="gray", linestyle="--", label="Recall goal")
plt.xlabel("Threshold")
plt.ylabel("Validation score")
plt.legend()
plt.show()
threshold_table

### Q7. Pick a threshold

Keep the rows of **threshold_table** whose **recall** is at least 0.70, and save the **highest** remaining threshold as **chosen_threshold**. The provided lines then check it once on the test set.

**Hint:** Filter with a Boolean condition, as in Lab 01, then take the **.max()** of the **threshold** column.

In [ ]:
meets_goal = threshold_table[...]
chosen_threshold = ...

test_pred = classifier.predict_proba(test[features])[:, 1] >= chosen_threshold
print("Chosen threshold:", chosen_threshold)
print("Test precision:", round(metrics.precision_score(y_test_c, test_pred), 3),
      "| Test recall:", round(metrics.recall_score(y_test_c, test_pred), 3))

In [ ]:
grader.check("q7")

**Your response:** Why did we choose the threshold on the **validation** set rather than the test set? If coupons became very expensive, would you raise or lower the threshold?

> Write your answer here.

### Optional: ROC curve and AUC

An **ROC curve** shows recall against the false positive rate, $\frac{FP}{FP+TN}$, at every threshold. Its **AUC** (area under the curve) summarizes how well the model ranks refund orders above others: 0.5 is random guessing and 1.0 is perfect.

In [ ]:
metrics.RocCurveDisplay.from_predictions(y_val_c, val_probs, name="Logistic regression")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random guessing")
plt.legend()
plt.show()

## Wrap-up

In 2–3 sentences: Which metric would you report for the delivery-time model, and which for the refund model? How did cross-validation let you choose alpha without touching the test set?

> Write your answer here.

In [ ]:
grader.check_all()